在 Megatron-LM 中，overlap-grad-reduce（梯度归约重叠）的实现主要分布在以下几个核心模块和文件中：
1. 配置入口
该功能通常通过启动参数 --overlap-grad-reduce 传入。在代码层面，它会被映射到 DistributedDataParallelConfig 类的 overlap_grad_reduce 属性中。

2. 核心包装器：distributed_data_parallel

<font color='red'>这是实现通信与计算重叠的核心模型包装器。</font>其主要逻辑包括：
- 梯度分桶（Bucketing）：
  - 将模型的参数和梯度分配到连续的缓冲区（GradBuffer）中，并根据配置的 bucket_size 划分为多个小的 Bucket。这为细粒度的异步通信提供了基础。
- 反向传播 Hook：
  - 在模型的反向传播之后注册了 Hook。当某个数据桶（Bucket）内的所有参数梯度累加完成后，系统会立即为该桶启动异步的集合通信操作（如 reduce-scatter 或 all-reduce），而不是等待整个模型的反向传播结束。
- 同步完成：
  - 在 finish_grad_sync() 方法中，如果启用了 overlap_grad_reduce=True，系统会调用 wait() 方法来等待之前发出的异步通信句柄（communication_handle）执行完毕，确保优化器更新前梯度已完全同步。
  
3. 调度与协调层：finalize_model_grads
  - 该模块充当了所有并行模式下梯度同步的协调层。它会在训练循环的关键节点被调用，负责遍历所有的模型块并触发上述的 finish_grad_sync() 操作，以聚合等待所有的异步 all-reduce / reduce-scatter 通信。
  
4. 流水线调度：pipeline_parallel/schedules.py
  - 在流水线并行场景下，梯度的同步时机需要与微批次（micro-batch）的执行紧密结合。在 schedules.py 中（例如 forward_backward_no_pipelining 函数内），在所有微批次的反向传播完成后，会调用 config.finalize_model_grads_func([model]) 来统一执行跨数据并行副本的梯度同步操作。

# DistributedDataParallel
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/distributed/distributed_data_parallel.py#L10

## \_\_init\_\_

### 核心功能与定位
DistributedDataParallel 的主要职责是管理模型在数据并行维度上的梯度同步。
- 基本原理：
  - 在数据并行训练中，每个 GPU 都持有一份完整的模型副本。
    - <font color='red'>DDP 负责在反向传播计算出梯度后，通过 All-Reduce（或 Reduce-Scatter）操作将所有 GPU 上的梯度汇总并取平均，然后更新模型参数。</font>
- 代码定位：
  - 在你提供的文档中，DistributedDataParallel <font color='red'>继承自 _BaseDataParallel</font>，它不仅实现了基础的梯度同步，还包含了梯度累积（Gradient Accumulation）、混合精度训练以及通信重叠（Overlap）等高级特性。
  
### 关键特性详解（基于文档代码）

#### A. 梯度缓冲与桶化（Bucketing）

为了提高通信效率，DDP 通常不会在每一个参数计算出梯度后立即进行同步，而是将多个参数的梯度“打包”在一起传输。
  - Bucket Size：文档中的 bucket_size 参数定义了每个“包”的大小。<font color='green'>代码逻辑会根据 dp_group.size()（数据并行组的大小）动态调整这个值，以确保在大规模并行时，传输的数据块足够大，从而保持带宽饱和，避免因频繁的小数据包传输导致延迟瓶颈。</font>
  - 禁用桶化：如果 disable_bucketing 为 True，或者当前处于流水线并行（PP）的非首阶段（pp_rank > 0），则会强制将所有参数放入一个桶中。<font color='red'>这是因为非首阶段的通信通常不在关键路径上，或者在特定调度（如交错调度）下不需要细粒度的桶化。</font>
  
#### B. <font color='blue'>通信与计算重叠</font>

这是 DDP 提高性能的关键手段，旨在<font color='red'>利用计算的时间来隐藏通信的延迟。</font>
  - 梯度归约重叠：
    - 机制：当配置 overlap_grad_reduce 为 True 时，DDP 会将梯度划分为多个较小的桶（Buckets）。
    - 执行：<font color='red'>一旦某个桶内的梯度计算完成，它会立即异步启动 All-Reduce 通信，而此时 GPU 可以继续执行其他层的反向传播计算。</font>
       - 这通过 _allocate_buffers_for_parameters 中的 partition_buckets 和流（Stream）管理实现。
  - 参数聚合重叠：
    - 场景：在使用分布式优化器（Distributed Optimizer）时，优化器状态（如动量、方差）是分片存储的。
    - 机制：在前向传播（Forward Pass）之前，需要将分片的参数“收集”（All-Gather）起来。
      - 如果启用 overlap_param_gather，DDP 会将这个收集操作分解为多个阶段，并与前向计算重叠。代码中通过 next_param_gather_bucket_group 链表结构来管理这种依赖关系。
      
#### C. 混合精度与 FP8 支持

文档代码显示了对 FP8（Float8） 训练的原生支持，这是现代大模型训练的重要特性。
  - 数据类型处理：代码中专门处理了 Float8Tensor 的情况。由于 FP8 参数在逻辑上表现为高精度（如 BF16），但物理存储为 uint8，DDP 在构建梯度缓冲区（_ParamAndGradBuffer）时会进行特殊的类型映射和索引记录，以确保在加载检查点或进行通信时数据类型的一致性。
  
#### D. 专家并行（Expert Parallelism）支持

代码中区分了 dense_params（密集参数）和 expert_parallel_params（专家参数）。
  - 独立处理：专家并行（如 MoE 架构中的专家网络）通常使用不同的通信组（expt_dp_group）和不同的梯度缩放因子（expert_gradient_scaling_factor）。DDP 在这里为专家参数分配了独立的缓冲区（expert_parallel_buffers），以确保它们的梯度同步逻辑与常规参数隔离，避免不必要的通信开销。
  
### 初始化流程解析
根据文档中的 __init__ 方法，DDP 的构建过程如下：

1. <font color='green'>配置解析</font>：接收 TransformerConfig 和 DistributedDataParallelConfig，确定通信组、桶大小等。
2. <font color='green'>进程组设</font>置：初始化各种并行维度的通信组（Data Parallel, Tensor Parallel, Pipeline Parallel 等）。
3. <font color='green'>参数分组</font>：<font color='red'>遍历模型参数，将它们分为需要梯度更新的参数、专家参数等，并建立参数到名称的映射。</font>
4. <font color='green'>缓冲区内</font>存分配：调用 _allocate_buffers_for_parameters。
  - <font color='red'>这是最复杂的一步，它根据数据类型（Dense/Expert）、精度（FP8/BF16）和梯度缩放因子，创建连续的内存缓冲区（Contiguous Buffers）来存储梯度。</font>
5. <font color='green'>钩子注册</font>：注册反向传播钩子（Backward Hook）和前向传播钩子（Forward Hook）。
  - Backward Hook：<font color='red'>用于触发梯度归约（Reduce）。</font>
  - Forward Hook：如果启用了 overlap_param_gather，<font color='red'>用于在前向计算期间触发参数的收集（All-Gather）。</font>
  
4. 总结
- DistributedDataParallel 在 Megatron-LM 中不仅仅是一个简单的“梯度同步器”，
  - 它是一个高度优化的分布式训练引擎。
      - <font color='red'>它通过桶化（Bucketing）、流式异步通信以及对FP8和专家并行的深度适配，最大限度地压榨了硬件（GPU 和 NVLink/InfiniBand）的性能，是支撑千亿级大模型高效训练的基石。</font>



In [ ]:
# Copyright (c) 2024, NVIDIA CORPORATION. All rights reserved.

import logging
from contextlib import contextmanager
from typing import Optional

import torch

from ..config_logger import has_config_logger_enabled, log_config_to_disk
from ..fp8_utils import is_float8tensor, post_all_gather_processing
from ..process_groups_config import ProcessGroupCollection
from ..transformer.cuda_graphs import is_graph_capturing
from ..transformer.transformer_config import TransformerConfig
from ..utils import log_single_rank
from .data_parallel_base import _BaseDataParallel
from .distributed_data_parallel_config import DistributedDataParallelConfig
from .param_and_grad_buffer import _ParamAndGradBuffer, partition_buckets

logger = logging.getLogger(__name__)


class DistributedDataParallel(_BaseDataParallel):
    """
    DDP wrapper which stores grads in contiguous buffers. Also has option of overlapping
    communication with backprop computation by breaking up full model's gradients into smaller
    buckets and running all-reduce / reduce-scatter on each bucket asynchronously. This class
    also provides the option to do the gradient accumulation in a type other than the param type
    (e.g., fp32 for a bf16 model).

    Args:
        config: Transformer config object.
        ddp_config: DistributedDataParallel config object.
        module: Underlying model.
        disable_bucketing: If true, force assign all parameters to a single bucket. If false,
            use standard bucketing policy: assign parameters to smaller buckets and all-reduce
            per bucket _if_ overlap_grad_reduce is True and pp_rank is 0.
        pg_collection: Optional unified process group for distributed training.

    """

    def __init__(
        self,
        config: TransformerConfig,
        ddp_config: DistributedDataParallelConfig,
        module: torch.nn.Module,
        disable_bucketing: bool = False,
        pg_collection: Optional[ProcessGroupCollection] = None,
    ):
        super().__init__(config=config, module=module)
        if has_config_logger_enabled(config):
            log_config_to_disk(config, locals(), prefix=type(self).__name__)

        # If bucket_size is not provided as an input, use sane default.
        # If using very large dp_sizes, make buckets larger to ensure that chunks used in NCCL
        # ring-reduce implementations are large enough to remain bandwidth-bound rather than
        # latency-bound.
        # Setup process groups, handling both None and provided pg_collection values.
        '''
        分布式通信组的初始化与赋值:
        
        在 Megatron-LM 这种混合并行框架中，GPU 之间不仅仅是简单的数据并行，还涉及张量并行（TP）、专家并行（EP）等
        '''
        process_group_dict = ProcessGroupCollection.setup_process_groups_for_ddp(
            pg_collection, config, ddp_config
        )

        # If bucket_size is not provided as an input, use sane default based on dp_group size.
        dp_group = process_group_dict['dp_group']
        if ddp_config.bucket_size is None:
            '''
            如果没有手动设置，它会执行一套基于数据并行组大小的智能默认策略。
            
            桶的大小被设置为 4000万 和 100万 × DP组进程数 两者中的较大值。
            
            技术原理：
             - 带宽约束 vs. 延迟约束：现代集合通信库（如 NCCL）在实现 Ring-Reduce 算法时，
               如果数据块太小，通信时间主要消耗在建立连接和传输控制信息上（延迟约束），
               导致带宽利用率低；如果数据块足够大，传输时间主要消耗在搬运数据上（带宽约束），效率最高。
             - 大模型适配：当数据并行度（dp_group.size()）非常大时（例如几百上千张卡），
               为了保证通信效率，必须让每个“桶”里的参数足够多，以确保 Reduce 操作的数据块足够大，
               从而维持在带宽受限的高效区间运行。

            '''
            ddp_config.bucket_size = max(40000000, 1000000 * dp_group.size())
            
        # Set bucket_size to infinity if overlap_grad_reduce is False.
        '''
        如果用户关闭了梯度归约重叠（overlap_grad_reduce=False），
        则将桶大小设置为 None（通常在内部会被解释为“无穷大”或“所有参数放入一个桶”）。
        
        技术原理：通信与计算重叠（Overlap）依赖于将梯度切分为多个小桶，以便在反向传播过程中分批触发通信。如果不需要重叠，
                那么将所有梯度合并为一次大的通信操作，可以减少通信启动的开销（Kernel Launch Overhead）。
        '''
        if not ddp_config.overlap_grad_reduce:
            ddp_config.bucket_size = None

        self.ddp_config = ddp_config
        log_single_rank(
            logger,
            logging.INFO,
            f'Setting up DistributedDataParallel with config {self.ddp_config}',
        )

        # Assign all required process groups
        self.dp_group = process_group_dict['dp_group']
        self.dp_cp_group = process_group_dict['dp_cp_group']
        self.intra_dp_cp_group = process_group_dict['intra_dp_cp_group']
        self.expt_dp_group = process_group_dict['expt_dp_group']
        self.intra_expt_dp_group = process_group_dict['intra_expt_dp_group']
        self.tp_group = process_group_dict['tp_group']
        self.pp_group = process_group_dict['pp_group']
        self.ep_group = process_group_dict['ep_group']

        # Set inter_dist_opt_group if multiple optimizer instances
        if self.ddp_config.num_distributed_optimizer_instances > 1:
            self.inter_dist_opt_group = process_group_dict['inter_dist_opt_group']

        # Turn off bucketing if we are on a pipeline stage that is not the first (since
        # data-parallel communication on these stages is not on the critical path), or if
        # disable_bucketing is True (e.g., we might not want to break up model parameters
        # into buckets for model chunks after the first in the interleaved schedule).
        self.bucket_size = self.ddp_config.bucket_size
        self.force_all_reduce = False
        if isinstance(self.pp_group, list):
            pp_rank = self.pp_group[0].rank()
        else:
            pp_rank = self.pp_group.rank()
        if disable_bucketing or pp_rank > 0:
            self.bucket_size = None

        self.param_to_bucket_group = {}

        # Group parameters by their gradient type.
        param_to_name = {}
        dense_params = []
        expert_parallel_params = []
        self.params_with_grad = []
        for name, param in self.module.named_parameters():
            if not param.requires_grad:
                continue

            # Track params with grad to enable direct setting
            # of param.grad_added_to_main_grad
            self.params_with_grad.append(param)

            param.grad_added_to_main_grad = False
            param_to_name[param] = name

            if getattr(param, 'allreduce', True):
                dense_params.append((param, name))
            else:
                expert_parallel_params.append((param, name))

        def _allocate_buffers_for_parameters(
            input_params, data_parallel_group, gradient_scaling_factor
        ):
            param_and_grad_dtype_to_params = {}
            param_and_grad_dtype_to_offsets = {}
            param_and_grad_dtype_to_indices = {}

            # Group parameters by their gradient type.
            for param, param_name in input_params:
                assert param.requires_grad

                param_dtype = param.dtype
                if is_float8tensor(param):
                    # Currently TE's Float8Tensor is a wrapper of torch.Tensor. It has a "fake"
                    # dtype (usually a higher precision dtype such as bfloat16), but its actual
                    # data is stored in the form of a torch uint8 tensor within the Float8Tensor's
                    # ".data" attribute. Therefore, when creating the param buffer for fp8 params,
                    # it is necessary to use torch.uint8, not the "fake" dtype got from
                    # "param.dtype".
                    param_dtype = torch.uint8
                grad_dtype = torch.float if self.ddp_config.grad_reduce_in_fp32 else param.dtype

                params = param_and_grad_dtype_to_params.get((param_dtype, grad_dtype), [])
                params.append((param, param_name))
                param_and_grad_dtype_to_params[(param_dtype, grad_dtype)] = params

                # Get the index of each param among the params with same dtype, if a param is fp8,
                # use its "fake" high precision dtype to find which params have same dtype with it.
                # For example:
                #     Case 1:
                #         params = [p1(bf16), p2(bf16), p3(bf16), p4(bf16)]
                #         param_and_grad_dtype_to_indices = {
                #             (torch.bfloat16, torch.float32): [0, 1, 2, 3],
                #         }
                #     Case 2:
                #         params = [p1(bf16), p2(fp8), p3(fp8), p4(bf16)]
                #         param_and_grad_dtype_to_indices = {
                #             (torch.bfloat16, torch.float32): [0, 3],
                #             (torch.uint8, torch.float32): [1, 2],
                #         }
                # We need these indices to load a non-native-fp8 checkpoint in native-fp8 mode.
                offset = param_and_grad_dtype_to_offsets.get((param.dtype, grad_dtype), 0)
                param_and_grad_dtype_to_offsets[(param.dtype, grad_dtype)] = offset + 1
                indices = param_and_grad_dtype_to_indices.get((param_dtype, grad_dtype), [])
                indices.append(offset)
                param_and_grad_dtype_to_indices[(param_dtype, grad_dtype)] = indices

            if not config.calculate_per_token_loss:
                target_gradient_scaling_factor = 1.0 / self.dp_cp_group.size()
                if self.ddp_config.average_in_collective:
                    if self.ddp_config.num_distributed_optimizer_instances == 1:
                        # Collective is averaging gradients in collective with data_parallel_group.
                        assert (
                            gradient_scaling_factor / data_parallel_group.size()
                            == target_gradient_scaling_factor
                        )
                    else:
                        # For non-expert parameters, gradient_scaling_factor is 1.
                        # For expert parameters, gradient_scaling_factor is edp_size/dp_size.
                        assert (gradient_scaling_factor == 1) or (
                            gradient_scaling_factor
                            == (self.expt_dp_group.size() / self.dp_cp_group.size())
                        )
                else:
                    assert gradient_scaling_factor == target_gradient_scaling_factor

            # Allocate the grad buffers and map the grads.
            buffers = []
            pg_collection = ProcessGroupCollection()
            pg_collection.tp = self.tp_group
            pg_collection.dp_cp = self.dp_cp_group
            for (param_dtype, grad_dtype), params in param_and_grad_dtype_to_params.items():
                buffers.append(
                    _ParamAndGradBuffer(
                        self.ddp_config,
                        param_dtype,
                        grad_dtype,
                        params,
                        data_parallel_group,
                        self.bucket_size,
                        param_to_name,
                        gradient_scaling_factor,
                        param_and_grad_dtype_to_indices[(param_dtype, grad_dtype)],
                        self.ddp_config.nccl_ub,
                        pg_collection,
                    )
                )

            # In some scenarios, we want to put buckets from different buffers into a group so that
            # their communication can be aggregated. For example, when there are both fp8 buffers
            # and bf16 buffers in the model and vpp is enabled, each model chunk will have an fp8
            # bucket and a bf16 bucket, which doubles the number of communication kernels, and
            # because of the use of CUDA_DEVICE_MAX_CONNECTIONS=1, having multiple back-to-back
            # communications will prevent the overlap of the communication kernels with computation
            # kernels.
            # If bucketing is explicitly disabled, then put all buckets in a buffer into a single
            # bucket group.
            bucket_groups = partition_buckets(buffers, force_single_bucket_group=disable_bucketing)

            if self.ddp_config.num_distributed_optimizer_instances > 1:
                assert (
                    self.ddp_config.use_distributed_optimizer
                ), 'Partial DistOpt cannot be used without DistOpt'
                communication_stream = torch.cuda.Stream(device=torch.cuda.current_device())
                for bucket_group in bucket_groups:
                    bucket_group.inter_distributed_optimizer_instance_group = (
                        self.inter_dist_opt_group
                    )
                    bucket_group.communication_stream = communication_stream

            # Set `next_param_gather_bucket_group` for different bucket groups by iterating through
            # buckets in reverse order (since all-gathers happen in reverse order of buckets).
            # Note: overlap_param_gather covers both the distributed optimizer and the
            # layer-wise optimizer cases; the latter sets overlap_param_gather=True
            # without use_distributed_optimizer.
            if self.ddp_config.overlap_param_gather:
                num_bucket_groups = len(bucket_groups)
                for i in range(1, num_bucket_groups):
                    bucket_groups[num_bucket_groups - i].next_param_gather_bucket_group = (
                        bucket_groups[num_bucket_groups - i - 1]
                    )

            # Create map from param to bucket group, used in pre_hook.
            for bucket_group in bucket_groups:
                for bucket in bucket_group.buckets:
                    for param in bucket.params_list:
                        self.param_to_bucket_group[param] = bucket_group

            return buffers, bucket_groups

        if config.calculate_per_token_loss:
            assert (
                not self.ddp_config.average_in_collective
            ), "Cannot average in collective when calculating per-token loss!"
            gradient_scaling_factor = 1.0
            expert_gradient_scaling_factor = 1.0
        else:
            # The goal is to scale reduced gradients by 1/dp_size.
            # This can be achieved in two ways:
            #
            # Case 1: average_in_collective=True
            # - Non-expert parameters:
            #   1. No pre-scaling (gradient_scaling_factor=1.0)
            #   2. Do average reduction over dp group (equals to sum then divide by dp_size)
            #   3. Final result is scaled by 1/dp_size as desired
            #
            # - Expert parameters:
            #   1. Scale by edp_size/dp_size before reduction
            #   2. Do average reduction over edp group (equals to sum then divide by edp_size)
            #   3. Resulted scaling: (edp_size/dp_size) * (1/edp_size) = 1/dp_size as desired
            #   (edp_size = expert data parallel world size)
            #
            # Case 2: average_in_collective=False
            # - Both expert and non-expert parameters:
            #   1. Scale gradients by 1/dp_size before reduction
            #   2. Do sum reduction across data parallel ranks
            #   3. Final result is scaled by 1/dp_size as desired
            if self.ddp_config.average_in_collective:
                gradient_scaling_factor = 1.0
                expert_gradient_scaling_factor = self.expt_dp_group.size() / self.dp_cp_group.size()
            else:
                data_parallel_world_size = self.dp_cp_group.size()

                gradient_scaling_factor = 1.0 / data_parallel_world_size
                expert_gradient_scaling_factor = 1.0 / data_parallel_world_size

        # Allocate the param+grad buffers for dense params' grads.
        self.buffers, self.bucket_groups = _allocate_buffers_for_parameters(
            dense_params, self.intra_dp_cp_group, gradient_scaling_factor=gradient_scaling_factor
        )

        # Allocate separate param+grad buffers for expert parallel params' grads.
        self.expert_parallel_buffers, self.expert_parallel_bucket_groups = (
            _allocate_buffers_for_parameters(
                expert_parallel_params,
                self.intra_expt_dp_group,
                gradient_scaling_factor=expert_gradient_scaling_factor,
            )
        )

        # Delete references to weight_tensor if they exist since we don't want two parameter copies
        # if we re-mapped parameters (which happens when we use the distributed optimizer).
        # This is a temporary workaround around a TE bug that is fixed with
        # https://github.com/NVIDIA/TransformerEngine/pull/719.
        if self.ddp_config.use_distributed_optimizer:

            @torch.no_grad()
            def unmap_weight_tensor(m):
                if hasattr(m, 'weight_tensor'):
                    m.weight_tensor = None

            self.module.apply(unmap_weight_tensor)

        # Register backward hook.
        # Accumulation function for the gradients need to be stored so they
        # don't go out of scope.
        self.grad_accs = []
        for param in self.module.parameters():
            if param.requires_grad:
                # When delay_wgrad_compute is True and the param is marked with
                # skip_backward_post_hook, register the backward post hook for its module
                # instead of the param so that the wgrad accumulation and reduce will be performed
                # in backward_dw() method of the module instead of the hook of backward() method.
                # Otherwise, register the backward post hook for the param.
                if self.ddp_config.delay_wgrad_compute and getattr(
                    param, 'skip_backward_post_hook', False
                ):
                    for module in self.module.modules():
                        if hasattr(module, "register_wgrad_accumulation_and_reduce_hooks"):
                            for param_value in module.parameters():
                                if param is param_value:
                                    module.register_wgrad_accumulation_and_reduce_hooks(
                                        self._make_backward_post_hook(param)
                                    )
                                    break
                else:
                    # Expand so we get access to grad_fn.
                    param_tmp = param.expand_as(param)
                    # Get the gradient accumulator function.
                    grad_acc = param_tmp.grad_fn.next_functions[0][0]
                    grad_acc.register_hook(self._make_backward_post_hook(param))
                    self.grad_accs.append(grad_acc)

        # Note: overlap_param_gather covers both the distributed optimizer and the
        # layer-wise optimizer cases; the latter sets overlap_param_gather=True
        # without use_distributed_optimizer.
        self.use_forward_hook = self.ddp_config.overlap_param_gather
        self.remove_forward_pre_hook_handles = {}
        if self.use_forward_hook:
            self.enable_forward_pre_hook()
        self.overlap_param_gather_with_optimizer_step = False

    

######  _BaseDataParallel
- MegatronModule的内容见 2.1.1- MegatronModule



In [ ]:
from ..transformer.module import MegatronModule
from ..transformer.transformer_config import TransformerConfig


class _BaseDataParallel(MegatronModule):
    """A template class for DistributedDataParallel implementations."""

    def __init__(self, config: TransformerConfig, module: torch.nn.Module):
        super().__init__(config=config)
        self.module = module

    def forward(self, *inputs, **kwargs):
        """
        Calls the wrapped module's forward() method.
        """
        return self.module(*inputs, **kwargs)


## enable_forward_pre_hook

In [ ]:
    def enable_forward_pre_hook(self):
        """
        Enable forward pre-hooks needed for param all-gather overlap with forward compute.
        """
        assert self.use_forward_hook
        assert len(self.remove_forward_pre_hook_handles) == 0
        # Register forward pre-hook for all sub-modules.
        for module in self.module.modules():
            self.remove_forward_pre_hook_handles[module] = module.register_forward_pre_hook(
                self._make_forward_pre_hook()
            )


## disable_forward_pre_hook

In [ ]:

    def disable_forward_pre_hook(self, param_sync: bool = True):
        """
        Disable forward pre-hooks needed for param all-gather overlap with forward compute.
        Skip synchronous param all-gather if `param_sync` is False.
        """
        assert self.use_forward_hook
        # De-register forward pre-hook for all sub-modules.
        for module in self.module.modules():
            assert self.remove_forward_pre_hook_handles[module] is not None
            self.remove_forward_pre_hook_handles[module].remove()
            del self.remove_forward_pre_hook_handles[module]
        assert len(self.remove_forward_pre_hook_handles) == 0

        # Force synchronize parameters.
        if param_sync:
            self.start_param_sync(force_sync=True)


## _make_forward_pre_hook

In [ ]:

    def _make_forward_pre_hook(self):
        """
        Create a forward pre-hook to wait on all-gather handles when necessary (i.e.,
        when a module uses a parameter in a bucket with a still incomplete all-gather).
        """

        def hook(module, *unused):
            assert (
                self.use_forward_hook
            ), "Should use pre-hook only when overlap_param_gather is True"

            if is_graph_capturing():
                return

            # Make sure all parameters in this module have been all-gathered as necessary.
            for param in module.parameters(recurse=False):
                # Skip parameters without an associated buffer (such parameters have a
                # .requires_grad field equal to False).
                if param not in self.param_to_bucket_group:
                    continue
                assert param.requires_grad

                # If aligning param all-gather across pipeline stages, all-gather is dispatched
                # by start_param_sync calls in core/pipeline_parallelism/schedules.py.
                # If overlapping param all-gather with optimizer step, then all-gather has
                # already been dispatched in optimizer step.
                skip_next_bucket_dispatch = (
                    self.ddp_config.align_param_gather
                    or self.overlap_param_gather_with_optimizer_step
                )
                self.param_to_bucket_group[param].finish_param_sync(
                    skip_next_bucket_dispatch=skip_next_bucket_dispatch
                )

        return hook


## _make_backward_post_hook

In [ ]:

    def _make_backward_post_hook(self, param: torch.nn.Parameter):
        """
        Creates a backward post-hook to dispatch an all-reduce / reduce-scatter when
        ready (i.e., when all grads in a bucket have been computed in all microbatches
        in a batch).
        """

        def hook(*unused):
            if is_graph_capturing():
                return

            if param in self.param_to_bucket_group:
                assert param.requires_grad
                if self.ddp_config.overlap_grad_reduce:
                    assert (
                        param.grad is not None
                    ), 'param.grad being None is not safe when overlap_grad_reduce is True'
                if param.grad is not None and (
                    not param.grad_added_to_main_grad or getattr(param, 'zero_out_wgrad', False)
                ):
                    param.main_grad.add_(param.grad.data)
                param.grad = None

                if self.ddp_config.overlap_grad_reduce:
                    self.param_to_bucket_group[param].register_grad_ready(
                        param, self.force_all_reduce
                    )

        return hook


## no_sync

In [ ]:

    @contextmanager
    def no_sync(self):
        """
        Context manager that turns off gradient synchronization.
        """
        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.is_last_microbatch = False
        try:
            yield
        finally:
            for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
                bucket_group.is_last_microbatch = True


## start_param_sync

In [ ]:

    def start_param_sync(self, *unused, force_sync: bool = False, force_dispatch: bool = False):
        """
        Initiates param sync (all-gather) communication operations for all model parameters.

        By default, when overlap_param_gather is set to True, dispatches asynchronous communication
        calls; when overlap_param_gather is set to False, calls synchronous communication
        ops. Can override this default behavior using flags below.

        Args:
            force_sync (bool, optional): force synchronous collective regardless of
                other settings.
            force_dispatch (bool, optional): force dispatch regardless of other settings.
        """
        if not force_sync:
            # If overlapping param AG with optimizer step, AG should not be dispatched again
            # in forward_backward_step.
            if self.overlap_param_gather_with_optimizer_step and not force_dispatch:
                return

        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.start_param_sync(force_sync=force_sync)

            if not self.ddp_config.overlap_param_gather:
                # For MXFP8 params, we need to copy the all-gathered param data from the buffer to
                # the param.data, since param buffer is not mapped to model params for MXFP8 case.
                # The paramaters are cast from bf16 to MXFP8 during copy.
                # In the case of "overlap_param_gather=True", the param copy is done
                # in "finish_param_sync" stage after zeroing the shared gardient buffers.
                if self.ddp_config.reuse_grad_buf_for_mxfp8_param_ag:
                    for bucket in bucket_group.buckets:
                        is_bf16_weight_bucket = False
                        for param in bucket.params:
                            # Skip copying since bf16 weights in the mxfp8 model
                            # are already mapped to param.data.
                            if not is_float8tensor(param):
                                is_bf16_weight_bucket = True
                                break
                            param_start, param_end = bucket.param_to_index[param]
                            param_slice = bucket.param_data.view(-1)[param_start:param_end]
                            param.data.copy_(param_slice.view(param.data.shape))
                        if is_bf16_weight_bucket:
                            continue
                        # All-gathered params are not needed after being copied to param.data.
                        # Zero out the param buffer (shared with grad buffer) for gradient
                        # accumulation. We cannot zero out the entire grad buffer because one grad
                        # buffer may correspond to multiple param buffers. If we zero out the entire
                        # grad buffer, it would clear the data of those param buffers that have not
                        # yet completed AG.
                        bucket.param_data.zero_()
                else:
                    fp8_params = []
                    for bucket in bucket_group.buckets:
                        for param in bucket.params:
                            if is_float8tensor(param):
                                fp8_params.append(param)
                    if len(fp8_params) > 0:
                        post_all_gather_processing(fp8_params)


## start_grad_sync

In [ ]:

    def start_grad_sync(self, *unused):
        """
        Initiates grad sync (all-reduce or reduce-scatter) communication operations
        for all model gradients.

        When overlap_grad_reduce is set to True, dispatches asynchronous communication
        calls. When overlap_grad_reduce is set to False, calls synchronous
        communication ops.
        """
        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.start_grad_sync()


## finish_grad_sync

In [ ]:

    def finish_grad_sync(self, force_all_reduce: Optional[bool] = False):
        """
        Finishes grad sync (all-reduce or reduce-scatter) communication operations
        for all model gradients.

        When overlap_grad_reduce is set to True, waits for asynchronous communication
        calls to complete. When overlap_grad_reduce is set to False, calls synchronous
        communication ops.
        """
        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.finish_grad_sync(force_all_reduce=force_all_reduce)


In [ ]:

    def free_overlap_buffers(self):
        """Free overlap param-gather GPU buffers across all bucket groups."""
        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.free_overlap_buffers()

    def scale_gradients(self, scaling_factor: float):
        """Scale all gradients inside the buffers by `scaling_factor`."""
        for buffer in self.buffers + self.expert_parallel_buffers:
            buffer.scale_gradients(scaling_factor)



In [ ]:
    def zero_grad_buffer(self):
        """
        Zeros out all grad buffers. Needs to be called at the beginning of each
        training iteration.
        """
        if getattr(self.config, 'cuda_graph_impl', 'none') != 'transformer_engine':
            # Don't reset grad_added_to_main_grad when CUDA Graph is used.
            # Because in CUDA Graph it no longer has the opportunity to set it back
            # to True, and there will be a double-GA.
            for param in self.params_with_grad:
                param.grad_added_to_main_grad = False
        for buffer in self.buffers + self.expert_parallel_buffers:
            buffer.reset()
        for bucket_group in self.bucket_groups + self.expert_parallel_bucket_groups:
            bucket_group.reset()


In [ ]:

    def broadcast_params(self):
        """
        Syncs parameters across all DP ranks.
        """
        for param in self.module.parameters():
            is_expert_parallel = not getattr(param, 'allreduce', True)

            if is_expert_parallel:
                data_parallel_group = self.expt_dp_group
            else:
                data_parallel_group = self.dp_cp_group
            torch.distributed.broadcast(
                param.data,
                src=torch.distributed.get_global_rank(data_parallel_group, 0),
                group=data_parallel_group,
            )


In [ ]:

    def offload_grad_buffers(self, synchronize: bool = True, empty_cache: bool = True) -> None:
        """
        Free all grad_data tensors to release GPU memory.

        Uses storage().resize_(0) to release memory while keeping tensor views intact.
        All bucket.grad_data and param.main_grad views remain valid tensor objects
        (though accessing them during offload is undefined behavior).

        Args:
            synchronize: Whether to call torch.cuda.synchronize() before freeing.
            empty_cache: Whether to call torch.cuda.empty_cache() after freeing.
        """
        if synchronize:
            torch.cuda.synchronize()

        for buffer in self.buffers + self.expert_parallel_buffers:
            buffer.offload_to_cpu(move_params=False, move_grads=True)

        if empty_cache:
            torch.cuda.empty_cache()


In [ ]:

    def restore_grad_buffers(self, synchronize: bool = True) -> None:
        """
        Reallocate grad_data tensors on GPU.

        All existing views (bucket.grad_data, param.main_grad) automatically
        become valid again since they share the same storage. The grad_data
        is zeroed after reallocation.

        Args:
            synchronize: Whether to call torch.cuda.synchronize() after allocation.
        """
        for buffer in self.buffers + self.expert_parallel_buffers:
            buffer.reload_from_cpu(move_params=False, move_grads=True)

        if synchronize:
            torch.cuda.synchronize()